<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/LR42_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# LOGISTIC REGRESSION42_HANCOCK
# Same Logistic Regression as Hecktor
# SMOTE = Yes
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# Imports
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# Read Data
# ============================================================

with open('/content/drive/MyDrive/clinical_data.json') as f:
    clinical = pd.DataFrame(json.load(f))

with open('/content/drive/MyDrive/pathological_data.json') as f:
    pathology = pd.DataFrame(json.load(f))

df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print("Merged Dataset Shape:", df.shape)

# ============================================================
# HPV Labels
# ============================================================

df = df[
    df['hpv_association_p16'].isin(
        ['positive', 'negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative': 0,
    'positive': 1
})

print("\nClass Distribution")
print(df['HPV'].value_counts())

# ============================================================
# Build Modelling Dataset
# ============================================================

df_model = df[
[
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm',
    'HPV'
]
].copy()

# ============================================================
# Missing Values
# ============================================================

cat_cols = [
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status'
]

for col in cat_cols:

    df_model[col] = df_model[col].fillna(
        df_model[col].mode()[0]
    )

num_cols = [
    'age_at_initial_diagnosis',
    'days_to_first_treatment',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'infiltration_depth_in_mm'
]

for col in num_cols:

    df_model[col] = df_model[col].fillna(
        df_model[col].median()
    )

# ============================================================
# Label Encoding
# ============================================================

for col in cat_cols:

    df_model[col] = LabelEncoder().fit_transform(
        df_model[col].astype(str)
    )

# ============================================================
# Features and Target
# ============================================================

X = df_model.drop(columns=['HPV'])
y = df_model['HPV']

n_features = X.shape[1]

print("\nNumber of Features:", n_features)

# ============================================================
# Train/Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("\nTraining Patients:", len(y_train))
print("Test Patients:", len(y_test))

# ============================================================
# Standardisation
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# SMOTE
# ============================================================

smote = SMOTE(
    random_state=SEED
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ============================================================
# Logistic Regression Model
# ============================================================

model = LogisticRegression(
    random_state=SEED,
    max_iter=1000
)

model.fit(
    X_train_smote,
    y_train_smote
)

# ============================================================
# Predictions
# ============================================================

predicted = model.predict(
    X_test
)

probabilities = model.predict_proba(
    X_test
)[:,1]

# ============================================================
# Classification Report
# ============================================================

print()
print("Classification Report\n")

print(
    classification_report(
        y_test,
        predicted,
        digits=4
    )
)

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    predicted
)

print("Confusion Matrix")
print(cm)

# ============================================================
# Metrics
# ============================================================

accuracy = (
    (predicted == y_test).sum()
    /
    len(y_test)
)

bal_acc = balanced_accuracy_score(
    y_test,
    predicted
)

f1 = f1_score(
    y_test,
    predicted
)

auc = roc_auc_score(
    y_test,
    probabilities
)

print()
print(f"Accuracy:            {accuracy:.4f}")
print(f"Balanced Accuracy:   {bal_acc:.4f}")
print(f"F1-score:            {f1:.4f}")
print(f"AUC:                 {auc:.4f}")

# ============================================================
# MODEL SUMMARY
# ============================================================

print()
print("====================================")
print("LOGISTIC REGRESSION42_HANCOCK")
print("====================================")

print("Model : Logistic Regression")
print("Input Features :", n_features)
print("Max Iterations : 1000")
print("Random State : 42")
print("SMOTE : Yes")

print("Training Patients :", len(y_train))
print("Test Patients :", len(y_test))
print("Training after SMOTE :", len(y_train_smote))

print()

print("Final Results")
print("---------------------------")

print(f"Accuracy            : {accuracy:.4f}")
print(f"Balanced Accuracy   : {bal_acc:.4f}")
print(f"F1-score            : {f1:.4f}")
print(f"AUC                 : {auc:.4f}")

print()

print("Confusion Matrix")
print(cm)

print()
print("Analysis Complete.")

Mounted at /content/drive
Merged Dataset Shape: (763, 49)

Class Distribution
HPV
0    191
1    141
Name: count, dtype: int64

Number of Features: 25

Training Patients: 265
Test Patients: 67

After SMOTE:
HPV
0    152
1    152
Name: count, dtype: int64

Classification Report

              precision    recall  f1-score   support

           0     0.7568    0.7179    0.7368        39
           1     0.6333    0.6786    0.6552        28

    accuracy                         0.7015        67
   macro avg     0.6950    0.6983    0.6960        67
weighted avg     0.7052    0.7015    0.7027        67

Confusion Matrix
[[28 11]
 [ 9 19]]

Accuracy:            0.7015
Balanced Accuracy:   0.6983
F1-score:            0.6552
AUC:                 0.7720

LOGISTIC REGRESSION42_HANCOCK
Model : Logistic Regression
Input Features : 25
Max Iterations : 1000
Random State : 42
SMOTE : Yes
Training Patients : 265
Test Patients : 67
Training after SMOTE : 304

Final Results
---------------------------
Ac